# AQI Forecasting – Faisalabad
Run cells top to bottom in order.

## Step 1 – Mount Drive & Unzip Project

In [ ]:

# ── Step 1: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH   = '/content/drive/MyDrive/aqi_project.zip'
EXTRACT_TO = '/content/'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_TO)

os.chdir('/content/aqi_project')
print('Working directory:', os.getcwd())


## Step 2 – Install Dependencies

In [ ]:

# ── Step 2: Install all dependencies ──────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

# Core
pip('pandas', 'numpy', 'requests', 'python-dotenv', 'tenacity', 'pyarrow', 'joblib', 'scipy')

# ML
pip('scikit-learn')
pip('tensorflow')
pip('shap')

# Feature Store & Registry
pip('hopsworks')
pip('mlflow')

# Dashboard & API
pip('streamlit')
pip('fastapi', 'uvicorn[standard]', 'plotly', 'matplotlib', 'pyngrok')

print('\n✅ All packages installed successfully!')


## Step 3 – Verify Hopsworks Connection

In [ ]:

# ── Step 3: Verify Hopsworks Connection ───────────────────────────────────────
import hopsworks

project = hopsworks.login(
    host='c.app.hopsworks.ai',
    api_key_value='ra6IJ49mKNWLYMCw.rb7n19k8NuiGDMaYM5M725AYLlFkDIEmLHc2Ll7KhOB7SyyuIDz3lWMRn44GFyao',
    project='AQIprediction804'
)
print('✅ Connected to project:', project.name)


## Step 4 – Backfill Historical Data (run once)

In [ ]:

# ── Step 4: Backfill Historical Data (run once) ────────────────────────────────
import os
os.chdir('/content/aqi_project')
!python -m src.backfill_pipeline --start 2022-01-01 --end 2026-05-21


## Step 5 – Fetch Today's Live Data

In [ ]:

# ── Step 5: Fetch Today's Live Data ───────────────────────────────────────────
import os
os.chdir('/content/aqi_project')
!python -m src.feature_pipeline


## Step 6 – Train Models

In [ ]:

# ── Step 6: Train Models ───────────────────────────────────────────────────────
import os
os.chdir('/content/aqi_project')
!python -m src.training_pipeline


## Step 7 – Launch Streamlit Dashboard

In [ ]:

# ── Step 7: Launch Streamlit Dashboard ────────────────────────────────────────
import subprocess, time, shutil, os

os.chdir('/content/aqi_project')

# Install localtunnel (no account needed)
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

# Kill any existing streamlit
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

# Launch Streamlit
streamlit_path = shutil.which('streamlit')
subprocess.Popen(
    [streamlit_path, 'run', 'app/dashboard.py',
     '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)

# Create public URL
lt = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE)
url = lt.stdout.readline().decode().strip()
print('✅ Dashboard live at:', url)


## Step 8 – Launch FastAPI (optional)

In [ ]:

# ── Step 8: Launch FastAPI (optional) ─────────────────────────────────────────
import subprocess, time, shutil, os

os.chdir('/content/aqi_project')

# Install localtunnel (no account needed)
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

# Launch FastAPI
uvicorn_path = shutil.which('uvicorn')
subprocess.Popen(
    [uvicorn_path, 'app.api:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# Create public URL
lt = subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE)
url = lt.stdout.readline().decode().strip()
print('✅ API URL     :', url)
print('   /predict   :', url + '/predict')
print('   /docs      :', url + '/docs')
